In [1]:
# Goal: Use the DeFiLlama API to pull data on multiple tokens and use the TVL available via defillama (for free and without a key!) to create some exceptional interactive data visualizations with plotly.express (px)

In [3]:
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

import plotly.express as px

import requests
import urllib3
import json
import datetime as dt
import urllib
import urllib.parse
import time

from datetime import datetime, timedelta
from requests import Request, Session
from requests.exceptions import ConnectionError, Timeout, TooManyRedirects
from requests.packages.urllib3.exceptions import InsecureRequestWarning

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

import warnings
warnings.simplefilter('ignore')

In [5]:
# DFL Library import & API init:
from defillama import DefiLlama

llama = DefiLlama()

In [6]:
# help menu on 'llama' will show all of the built-in methods available via the dfl wrapper:
help(llama)

Help on DefiLlama in module defillama.defillama object:

class DefiLlama(builtins.object)
 |  DeFi Llama class to act as DeFi Llama's API client.
 |  All the requests can be made through this class.
 |  
 |  Methods defined here:
 |  
 |  __init__(self)
 |      Initialize the object
 |  
 |  get_all_protocols(self)
 |      Returns basic information on all listed protocols, their current
 |      TVL and the changes to it in the last hour/day/week.
 |      Endpoint: GET /protocols
 |      
 |      :return: JSON response
 |  
 |  get_batch_historical_prices(self, coins: str, searchWidth: str = '600')
 |      Get historical prices of tokens by contract address
 |  
 |  get_chain_dexs(self, chain, excludeTotalDataChart=True, excludeTotalDataChartBreakdown=True, dataType='dailyVolume')
 |      list all dexs filter by chain
 |  
 |  get_chain_options_dexs(self, chain, excludeTotalDataChart=True, excludeTotalDataChartBreakdown=True, dataType='dailyVolume')
 |      list all options dexs
 |  
 |

In [7]:
# Pull all historical data for Solana (entire chain); this will be used in our viz as a benchmark;

solana_data = llama.get_historical_tvl_chain('Solana')
solana_data

[{'date': 1616025600, 'tvl': 148988798},
 {'date': 1616112000, 'tvl': 153204289},
 {'date': 1616198400, 'tvl': 147690914},
 {'date': 1616284800, 'tvl': 151935325},
 {'date': 1616371200, 'tvl': 152981122},
 {'date': 1616457600, 'tvl': 162065403},
 {'date': 1616544000, 'tvl': 156101311},
 {'date': 1616630400, 'tvl': 144439504},
 {'date': 1616716800, 'tvl': 135610450},
 {'date': 1616803200, 'tvl': 147786656},
 {'date': 1616889600, 'tvl': 172583244},
 {'date': 1616976000, 'tvl': 189014228},
 {'date': 1617062400, 'tvl': 202157669},
 {'date': 1617148800, 'tvl': 209794033},
 {'date': 1617235200, 'tvl': 216074736},
 {'date': 1617321600, 'tvl': 236083655},
 {'date': 1617408000, 'tvl': 236307232},
 {'date': 1617494400, 'tvl': 236512420},
 {'date': 1617580800, 'tvl': 236284645},
 {'date': 1617667200, 'tvl': 272660066},
 {'date': 1617753600, 'tvl': 268791821},
 {'date': 1617840000, 'tvl': 279546575},
 {'date': 1617926400, 'tvl': 294784827},
 {'date': 1618012800, 'tvl': 331055199},
 {'date': 161809

In [8]:
# Since the data has no extra 'fluff' in the top of the raw json, can just load directly into a dataframe:
df_sol = pd.DataFrame(solana_data)

# The timestamps are in UNIX, so need to convert to datetime in a human-readable format, then set as the index and sort by the date:

df_sol['date'] = pd.to_datetime(df_sol['date'], unit='s')
df_sol.se